# Affine Body Diff Surface

In [166]:
import sympy as sp
import sys
sys.path.append('../')
from sympy import symbols, integrate, expand
from SymEigen import *
from project_dir import backend_source_dir

def output_path(name):
    return backend_source_dir('cuda') / f'affine_body/constitutions/sym/{name}.inl'

In [167]:
u,v = symbols('u v', real=True)
rho = symbols('rho', real=True)
X0 = Eigen.Vector('X0', 9)
xi = X0[0:3, 0]
xj = X0[3:6, 0]
xk = X0[6:9, 0]

In [168]:
def integrate_S(f):
    fv = integrate(f, (u, 0, 1-v))
    f = integrate(fv, (v, 0, 1))
    return f

I_dudv = integrate_S(1)

I_dudv

1/2

In [169]:
D = sp.zeros(3,3)
D[0:3, 0] = xi
D[0:3, 1] = xj
D[0:3, 2] = xk
D

Matrix([
[X0(0), X0(3), X0(6)],
[X0(1), X0(4), X0(7)],
[X0(2), X0(5), X0(8)]])

In [170]:
detD = det(D)
detD

X0(0)*X0(4)*X0(8) - X0(0)*X0(5)*X0(7) - X0(1)*X0(3)*X0(8) + X0(1)*X0(5)*X0(6) + X0(2)*X0(3)*X0(7) - X0(2)*X0(4)*X0(6)

In [171]:
V = I_dudv/3*detD

## Diff Volume

Say we have a shape energy:

$$
E = \kappa V S(q)
$$

$$
G = \kappa V \frac{\partial S}{\partial q}
$$

$$
\frac{\partial G}{\partial \bar{X}} = \kappa \frac{\partial S}{\partial q} \frac{\partial V}{\partial \bar{X}}^T
$$

In [172]:
pVpX = VecDiff(V, X0)
pVpX


Matrix([
[ X0(4)*X0(8)/6 - X0(5)*X0(7)/6],
[-X0(3)*X0(8)/6 + X0(5)*X0(6)/6],
[ X0(3)*X0(7)/6 - X0(4)*X0(6)/6],
[-X0(1)*X0(8)/6 + X0(2)*X0(7)/6],
[ X0(0)*X0(8)/6 - X0(2)*X0(6)/6],
[-X0(0)*X0(7)/6 + X0(1)*X0(6)/6],
[ X0(1)*X0(5)/6 - X0(2)*X0(4)/6],
[-X0(0)*X0(5)/6 + X0(2)*X0(3)/6],
[ X0(0)*X0(4)/6 - X0(1)*X0(3)/6]])

In [173]:

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction("__host__ __device__")
Gen.DisableLatexComment()
Cl = Gen.Closure(X0)
code_pVpX = Cl('pVpX0', pVpX)

To update the Volume from the $\bar{X}$:

In [174]:
code_V = Cl('V', V)

with open(output_path('diffV'), 'w') as f:
    content = f'''
{code_pVpX}
{code_V}'''
    # print(content)
    f.write(content)

## Diff Mass

An Affine Body mass $M$ has the following form:

In [175]:
M = sp.zeros(12,12)

sym_m = symbols('m')
M[0:3,0:3] = sp.eye(3) * sym_m

sym_La = sp.zeros(3,1)
for a in range(3):
    sym_La[a] = symbols(f'L{a}')

t = sp.zeros(rows=3, cols=9)
t[0,0:3] = sym_La.T
t[1,3:6] = sym_La.T
t[2,6:9] = sym_La.T
M[0:3,3:12] = t
M[3:12,0:3] = t.T

sym_Lab = sp.zeros(3,3)
    
for a in range(3):
    for b in range(3):
        sym_Lab[a,b] = symbols(f'L{a}{b}')
M[3:6,3:6] = sym_Lab
M[6:9,6:9] = sym_Lab
M[9:12,9:12] = sym_Lab

M

Matrix([
[ m,  0,  0,  L0,  L1,  L2,   0,   0,   0,   0,   0,   0],
[ 0,  m,  0,   0,   0,   0,  L0,  L1,  L2,   0,   0,   0],
[ 0,  0,  m,   0,   0,   0,   0,   0,   0,  L0,  L1,  L2],
[L0,  0,  0, L00, L01, L02,   0,   0,   0,   0,   0,   0],
[L1,  0,  0, L10, L11, L12,   0,   0,   0,   0,   0,   0],
[L2,  0,  0, L20, L21, L22,   0,   0,   0,   0,   0,   0],
[ 0, L0,  0,   0,   0,   0, L00, L01, L02,   0,   0,   0],
[ 0, L1,  0,   0,   0,   0, L10, L11, L12,   0,   0,   0],
[ 0, L2,  0,   0,   0,   0, L20, L21, L22,   0,   0,   0],
[ 0,  0, L0,   0,   0,   0,   0,   0,   0, L00, L01, L02],
[ 0,  0, L1,   0,   0,   0,   0,   0,   0, L10, L11, L12],
[ 0,  0, L2,   0,   0,   0,   0,   0,   0, L20, L21, L22]])

Now we will continue to define the remaining components of the matrix M and incorporate the symbols defined earlier.

$$
p = (1-u-v) x_i + u x_j + v x_k
$$

$$
\int \vec{F} \cdot d \vec{S} = \int (\vec{F} \cdot \vec{N}) p dudv
$$

In [176]:
N = (xj-xi).cross(xk-xi)
p = (1-u-v)*xi + u*xj + v*xk

In [177]:
m = rho*V

In [178]:
def L(a:int):
    Na = N[a]
    pa = p[a]
    Q = integrate_S(pa**2)
    return rho * Na * Q / 2

Las = sp.zeros(3,1)
for a in range(3):
    Las[a] = L(a)

Las[0]

rho*((-X0(1) + X0(4))*(-X0(2) + X0(8)) - (-X0(1) + X0(7))*(-X0(2) + X0(5)))*(X0(0)**2/12 + X0(0)*X0(3)/12 + X0(0)*X0(6)/12 + X0(3)**2/12 + X0(3)*X0(6)/12 + X0(6)**2/12)/2

In [179]:
def L(a:int, b:int):
    if(a==b):
        Na = N[a]
        pa = p[a]
        Q = integrate_S(pa**3)
        return rho * Na * Q / 3
    else:
        Na = N[a]
        pa = p[a]
        pb = p[b]
        Q = integrate_S(pa**2 * pb)
        return rho * Na * Q / 2

Labs = sp.zeros(3,3)

for a in range(3):
    for b in range(3):
        Labs[a,b] = L(a, b)

# Labs


For Kinetic Energy:

$$
E = \frac{1}{2} \|q-\tilde{q}\|_M
$$

$$
G = M dq
$$

where

$$
dq := q-\tilde{q}
$$

$$
\frac{\partial G}{\partial \bar{X}} = \frac{\partial (M dq)}{\partial \bar{X}}
$$


In [180]:
dq = Eigen.Vector('dq', 12)

G = M * dq
G

Matrix([
[      L0*dq(3) + L1*dq(4) + L2*dq(5) + dq(0)*m],
[      L0*dq(6) + L1*dq(7) + L2*dq(8) + dq(1)*m],
[    L0*dq(9) + L1*dq(10) + L2*dq(11) + dq(2)*m],
[  L0*dq(0) + L00*dq(3) + L01*dq(4) + L02*dq(5)],
[  L1*dq(0) + L10*dq(3) + L11*dq(4) + L12*dq(5)],
[  L2*dq(0) + L20*dq(3) + L21*dq(4) + L22*dq(5)],
[  L0*dq(1) + L00*dq(6) + L01*dq(7) + L02*dq(8)],
[  L1*dq(1) + L10*dq(6) + L11*dq(7) + L12*dq(8)],
[  L2*dq(1) + L20*dq(6) + L21*dq(7) + L22*dq(8)],
[L0*dq(2) + L00*dq(9) + L01*dq(10) + L02*dq(11)],
[L1*dq(2) + L10*dq(9) + L11*dq(10) + L12*dq(11)],
[L2*dq(2) + L20*dq(9) + L21*dq(10) + L22*dq(11)]])

In [181]:
G = G.subs({
    sym_m:m,
    #
    sym_La[0]:Las[0],
    sym_La[1]:Las[1],
    sym_La[2]:Las[2],
    #
    sym_Lab[0,0]:Labs[0,0],
    sym_Lab[0,1]:Labs[0,1],
    sym_Lab[0,2]:Labs[0,2],
    sym_Lab[1,0]:Labs[1,0],
    sym_Lab[1,1]:Labs[1,1],
    sym_Lab[1,2]:Labs[1,2],
    sym_Lab[2,0]:Labs[2,0],
    sym_Lab[2,1]:Labs[2,1],
    sym_Lab[2,2]:Labs[2,2]
})

In [182]:
pGpX = VecDiff(G, X0)

Cl = Gen.Closure(X0, dq, rho)
code_pGpX = Cl('pGpX0', pGpX)

In [183]:
Cl = Gen.Closure(X0, rho)
code_m = Cl('m', m)
code_La = Cl('La', Las)
code_Lab = Cl('Lab', Labs)

with open(output_path('diffM'), 'w') as f:
    content = f'''
{code_pGpX}
{code_m}
{code_La}
{code_Lab}'''
    # print(content)
    f.write(content)

## External Forces

In [184]:
f = Eigen.Vector('f', 3)
F = sp.zeros(12,1)

def L(a:int):
    Na = N[a]
    pa = p[a]
    Q = integrate_S(pa**2)
    return Na * Q / 2

Las = sp.zeros(3,1)
for a in range(3):
    Las[a] = L(a)

F[0:3,0] = f * V
F[3:6,0] = f[0] * Las
F[6:9,0] = f[1] * Las
F[9:12,0] = f[2] * Las

F

Matrix([
[                                    f(0)*(X0(0)*X0(4)*X0(8)/6 - X0(0)*X0(5)*X0(7)/6 - X0(1)*X0(3)*X0(8)/6 + X0(1)*X0(5)*X0(6)/6 + X0(2)*X0(3)*X0(7)/6 - X0(2)*X0(4)*X0(6)/6)],
[                                    f(1)*(X0(0)*X0(4)*X0(8)/6 - X0(0)*X0(5)*X0(7)/6 - X0(1)*X0(3)*X0(8)/6 + X0(1)*X0(5)*X0(6)/6 + X0(2)*X0(3)*X0(7)/6 - X0(2)*X0(4)*X0(6)/6)],
[                                    f(2)*(X0(0)*X0(4)*X0(8)/6 - X0(0)*X0(5)*X0(7)/6 - X0(1)*X0(3)*X0(8)/6 + X0(1)*X0(5)*X0(6)/6 + X0(2)*X0(3)*X0(7)/6 - X0(2)*X0(4)*X0(6)/6)],
[ f(0)*((-X0(1) + X0(4))*(-X0(2) + X0(8)) - (-X0(1) + X0(7))*(-X0(2) + X0(5)))*(X0(0)**2/12 + X0(0)*X0(3)/12 + X0(0)*X0(6)/12 + X0(3)**2/12 + X0(3)*X0(6)/12 + X0(6)**2/12)/2],
[f(0)*(-(-X0(0) + X0(3))*(-X0(2) + X0(8)) + (-X0(0) + X0(6))*(-X0(2) + X0(5)))*(X0(1)**2/12 + X0(1)*X0(4)/12 + X0(1)*X0(7)/12 + X0(4)**2/12 + X0(4)*X0(7)/12 + X0(7)**2/12)/2],
[ f(0)*((-X0(0) + X0(3))*(-X0(1) + X0(7)) - (-X0(0) + X0(6))*(-X0(1) + X0(4)))*(X0(2)**2/12 + X0(2)*X0(5)/12 + 

In [185]:

Cl = Gen.Closure(X0, f)
code_F = Cl('F', F)

diffF = VecDiff(F, X0)
code_diffF = Cl('diffF', diffF)

with open(output_path('abd_force'), 'w') as f:
    content = f'''
{code_F}
{code_diffF}
'''
    # print(content)
    f.write(content)

## Jacobi Mapping

When we have a vertex-based gradient $G(x)$, we need to map to the body-based gradient $G_{12}$.

$$
G_{12} = J^T(\bar{X}) G(x)
$$

For those $G_3 = G_3(X)$ (not related to $\bar{X}$), we have:

$$
\frac{\partial G_{12}}{\partial \bar{X}} = \frac{\partial J^T(\bar{X}) G(x)}{\partial \bar{X}}

= \frac{\partial J^T(\bar{X}) G}{\partial \bar{X}} + J^T(\bar{X}) \frac{\partial G}{\partial x} \frac{\partial {J(\bar{X})q}}{\partial \bar{X}}
$$

In [186]:
x0 = Eigen.Vector('X0', 3)
J = sp.Matrix.zeros(3,12)
J[0:3,0:3] = sp.eye(3)
J[0,3:6] = x0.T
J[1,6:9] = x0.T
J[2,9:12] = x0.T
J

Matrix([
[1, 0, 0, X0(0), X0(1), X0(2),     0,     0,     0,     0,     0,     0],
[0, 1, 0,     0,     0,     0, X0(0), X0(1), X0(2),     0,     0,     0],
[0, 0, 1,     0,     0,     0,     0,     0,     0, X0(0), X0(1), X0(2)]])

In [187]:
q = Eigen.Vector('q', 12)
x = J * q
x

Matrix([
[  X0(0)*q(3) + X0(1)*q(4) + X0(2)*q(5) + q(0)],
[  X0(0)*q(6) + X0(1)*q(7) + X0(2)*q(8) + q(1)],
[X0(0)*q(9) + X0(1)*q(10) + X0(2)*q(11) + q(2)]])

In [188]:
G3 = Eigen.Vector('G3', 3)
H3x3 = Eigen.Matrix('H3x3', 3, 3)
G12left = J.T * G3

In [189]:
pJGpx0 = VecDiff(G12left, x0)
pJGpx0

Matrix([
[    0,     0,     0],
[    0,     0,     0],
[    0,     0,     0],
[G3(0),     0,     0],
[    0, G3(0),     0],
[    0,     0, G3(0)],
[G3(1),     0,     0],
[    0, G3(1),     0],
[    0,     0, G3(1)],
[G3(2),     0,     0],
[    0, G3(2),     0],
[    0,     0, G3(2)]])

In [190]:
pJqpX0 = VecDiff(x, x0)
pJqpX0

Matrix([
[q(3),  q(4),  q(5)],
[q(6),  q(7),  q(8)],
[q(9), q(10), q(11)]])

In [191]:
pG12px0 = pJGpx0 + J.T * H3x3 * pJqpX0
pG12px0

Matrix([
[                          H3x3(0,0)*q(3) + H3x3(0,1)*q(6) + H3x3(0,2)*q(9),                           H3x3(0,0)*q(4) + H3x3(0,1)*q(7) + H3x3(0,2)*q(10),                           H3x3(0,0)*q(5) + H3x3(0,1)*q(8) + H3x3(0,2)*q(11)],
[                          H3x3(1,0)*q(3) + H3x3(1,1)*q(6) + H3x3(1,2)*q(9),                           H3x3(1,0)*q(4) + H3x3(1,1)*q(7) + H3x3(1,2)*q(10),                           H3x3(1,0)*q(5) + H3x3(1,1)*q(8) + H3x3(1,2)*q(11)],
[                          H3x3(2,0)*q(3) + H3x3(2,1)*q(6) + H3x3(2,2)*q(9),                           H3x3(2,0)*q(4) + H3x3(2,1)*q(7) + H3x3(2,2)*q(10),                           H3x3(2,0)*q(5) + H3x3(2,1)*q(8) + H3x3(2,2)*q(11)],
[G3(0) + H3x3(0,0)*X0(0)*q(3) + H3x3(0,1)*X0(0)*q(6) + H3x3(0,2)*X0(0)*q(9),         H3x3(0,0)*X0(0)*q(4) + H3x3(0,1)*X0(0)*q(7) + H3x3(0,2)*X0(0)*q(10),         H3x3(0,0)*X0(0)*q(5) + H3x3(0,1)*X0(0)*q(8) + H3x3(0,2)*X0(0)*q(11)],
[        H3x3(0,0)*X0(1)*q(3) + H3x3(0,1)*X0(1)*q(6) + H3x3(0,2

In [ ]:
Cl = Gen.Closure(x0, G3, H3x3, q)
code_pG12px0 = Cl('pG12px0', pG12px0)

with open(output_path('diffG12'), 'w') as f:
    content = f'''
{code_pG12px0}'''
    # print(content)
    f.write(content)

In [193]:
x0 = X0[0:3, 0]
x1 = X0[3:6, 0]
x2 = X0[6:9, 0]

A = ((x1-x0).cross(x2-x0)).norm()/2
A


sqrt(((-X0(0) + X0(3))*(-X0(1) + X0(7)) - (-X0(0) + X0(6))*(-X0(1) + X0(4)))**2 + ((X0(0) - X0(3))*(X0(2) - X0(8)) - (X0(0) - X0(6))*(X0(2) - X0(5)))**2 + ((-X0(1) + X0(4))*(-X0(2) + X0(8)) - (-X0(1) + X0(7))*(-X0(2) + X0(5)))**2)/2

In [194]:
diffA = VecDiff(A, X0)

Cl = Gen.Closure(X0)
code_A = Cl('A', A)
code_diffA = Cl('dAdx0', diffA)

with open(output_path('diffArea'), 'w') as f:
    content = f'''
{code_A}
{code_diffA}'''
    # print(content)
    f.write(content)